In [1]:
data = [(1,"2025-01-01","Laptop","Electronics","North",50000,1),
        (2,"2025-01-02","Phone","Electronics","South",30000,2),
        (3,"2025-01-03","Table","Furniture","East",15000,1),
        (4,"2025-01-04","Chair","Furniture","West",10000,4),
        (5,"2025-01-05","Headphones","Electronics","North",20000,3),
        (6,"2025-01-06","Sofa","Furniture","South",25000,1),
        (7,"2025-01-07","Camera","Electronics","East",35000,2)
        ]

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark  = SparkSession.builder.appName("PySparkModule2").getOrCreate()

In [4]:
spark

In [5]:
data

[(1, '2025-01-01', 'Laptop', 'Electronics', 'North', 50000, 1),
 (2, '2025-01-02', 'Phone', 'Electronics', 'South', 30000, 2),
 (3, '2025-01-03', 'Table', 'Furniture', 'East', 15000, 1),
 (4, '2025-01-04', 'Chair', 'Furniture', 'West', 10000, 4),
 (5, '2025-01-05', 'Headphones', 'Electronics', 'North', 20000, 3),
 (6, '2025-01-06', 'Sofa', 'Furniture', 'South', 25000, 1),
 (7, '2025-01-07', 'Camera', 'Electronics', 'East', 35000, 2)]

In [6]:
columns = ["OrderID", "OrderDate", "Product", "Category", "Region", "SalesAmount", "Quantity"]  

In [7]:
df = spark.createDataFrame(data,columns)

In [8]:
df.show()

+-------+----------+----------+-----------+------+-----------+--------+
|OrderID| OrderDate|   Product|   Category|Region|SalesAmount|Quantity|
+-------+----------+----------+-----------+------+-----------+--------+
|      1|2025-01-01|    Laptop|Electronics| North|      50000|       1|
|      2|2025-01-02|     Phone|Electronics| South|      30000|       2|
|      3|2025-01-03|     Table|  Furniture|  East|      15000|       1|
|      4|2025-01-04|     Chair|  Furniture|  West|      10000|       4|
|      5|2025-01-05|Headphones|Electronics| North|      20000|       3|
|      6|2025-01-06|      Sofa|  Furniture| South|      25000|       1|
|      7|2025-01-07|    Camera|Electronics|  East|      35000|       2|
+-------+----------+----------+-----------+------+-----------+--------+



In [13]:
df

DataFrame[OrderID: bigint, OrderDate: string, Product: string, Category: string, Region: string, SalesAmount: bigint, Quantity: bigint]

In [16]:
from pyspark.sql.functions  import * 

In [17]:
df.groupBy("Category").agg(sum(col("SalesAmount"))).show()

+-----------+----------------+
|   Category|sum(SalesAmount)|
+-----------+----------------+
|Electronics|          135000|
|  Furniture|           50000|
+-----------+----------------+



In [18]:
df.groupBy("Category").sum("SalesAmount").show() 

+-----------+----------------+
|   Category|sum(SalesAmount)|
+-----------+----------------+
|Electronics|          135000|
|  Furniture|           50000|
+-----------+----------------+



In [19]:
df.groupBy("Category").agg(sum(col("SalesAmount")).alias("TotalRevenue")).show()

+-----------+------------+
|   Category|TotalRevenue|
+-----------+------------+
|Electronics|      135000|
|  Furniture|       50000|
+-----------+------------+



In [21]:
df = df.withColumn("revenue",col("SalesAmount")*col("Quantity"))

In [22]:
df.show()

+-------+----------+----------+-----------+------+-----------+--------+-------+
|OrderID| OrderDate|   Product|   Category|Region|SalesAmount|Quantity|revenue|
+-------+----------+----------+-----------+------+-----------+--------+-------+
|      1|2025-01-01|    Laptop|Electronics| North|      50000|       1|  50000|
|      2|2025-01-02|     Phone|Electronics| South|      30000|       2|  60000|
|      3|2025-01-03|     Table|  Furniture|  East|      15000|       1|  15000|
|      4|2025-01-04|     Chair|  Furniture|  West|      10000|       4|  40000|
|      5|2025-01-05|Headphones|Electronics| North|      20000|       3|  60000|
|      6|2025-01-06|      Sofa|  Furniture| South|      25000|       1|  25000|
|      7|2025-01-07|    Camera|Electronics|  East|      35000|       2|  70000|
+-------+----------+----------+-----------+------+-----------+--------+-------+



In [23]:
df.groupBy("Category").agg(sum(col("revenue")).alias("TotalRevenue")).show()

+-----------+------------+
|   Category|TotalRevenue|
+-----------+------------+
|Electronics|      240000|
|  Furniture|       80000|
+-----------+------------+



In [24]:
df.groupBy("Region").agg(countDistinct("Category").alias("UniqueCategory")).show()

+------+--------------+
|Region|UniqueCategory|
+------+--------------+
| South|             2|
|  East|             2|
|  West|             1|
| North|             1|
+------+--------------+



In [25]:
df.groupBy("OrderDate").sum("revenue").show()

+----------+------------+
| OrderDate|sum(revenue)|
+----------+------------+
|2025-01-01|       50000|
|2025-01-02|       60000|
|2025-01-03|       15000|
|2025-01-04|       40000|
|2025-01-05|       60000|
|2025-01-06|       25000|
|2025-01-07|       70000|
+----------+------------+



In [26]:
df

DataFrame[OrderID: bigint, OrderDate: string, Product: string, Category: string, Region: string, SalesAmount: bigint, Quantity: bigint, revenue: bigint]

In [27]:
df= df.withColumn("OrderDate",to_date("OrderDate"))

In [28]:
df

DataFrame[OrderID: bigint, OrderDate: date, Product: string, Category: string, Region: string, SalesAmount: bigint, Quantity: bigint, revenue: bigint]

In [29]:
df.groupBy(month("OrderDate")).sum("revenue").show()

+----------------+------------+
|month(OrderDate)|sum(revenue)|
+----------------+------------+
|               1|      320000|
+----------------+------------+



In [30]:
revenue_df = df.groupBy("Region").agg(sum("revenue").alias("total_revenue"))

In [31]:
revenue_df.filter(col("total_revenue") > 75000).show()

+------+-------------+
|Region|total_revenue|
+------+-------------+
| North|       110000|
| South|        85000|
|  East|        85000|
+------+-------------+



In [41]:
orders = [(1,101,"laptop",50000),
          (2,102,"phone",30000),
          (3,103,"table",15000),
          (4,104,"chair",10000),
          (5,105,"headphones",20000),
          (7,107,"camera",35000)
          ]
order_columns = ["OrderID", "CustomerID", "Product", "SalesAmount"]

In [51]:
customer = [(101,"Alice","North"),
            (102,"Bob","South"),
            (103,"Charlie","East"),
            (104,"David","West"),
            (105,"Eve","North"),
            (108,"Grace","East")
            ]
customer_columns = ["CustomerID", "CustomerName", "Region"]

In [52]:
orders_df = spark.createDataFrame(orders, order_columns)

In [53]:
customer_df = spark.createDataFrame(customer, customer_columns)

In [54]:
orders_df.show()

+-------+----------+----------+-----------+
|OrderID|CustomerID|   Product|SalesAmount|
+-------+----------+----------+-----------+
|      1|       101|    laptop|      50000|
|      2|       102|     phone|      30000|
|      3|       103|     table|      15000|
|      4|       104|     chair|      10000|
|      5|       105|headphones|      20000|
|      7|       107|    camera|      35000|
+-------+----------+----------+-----------+



In [55]:
customer_df.show()

+----------+------------+------+
|CustomerID|CustomerName|Region|
+----------+------------+------+
|       101|       Alice| North|
|       102|         Bob| South|
|       103|     Charlie|  East|
|       104|       David|  West|
|       105|         Eve| North|
|       108|       Grace|  East|
+----------+------------+------+



In [56]:
orders_df.join(customer_df,"CustomerID","inner").show()

+----------+-------+----------+-----------+------------+------+
|CustomerID|OrderID|   Product|SalesAmount|CustomerName|Region|
+----------+-------+----------+-----------+------------+------+
|       101|      1|    laptop|      50000|       Alice| North|
|       102|      2|     phone|      30000|         Bob| South|
|       103|      3|     table|      15000|     Charlie|  East|
|       104|      4|     chair|      10000|       David|  West|
|       105|      5|headphones|      20000|         Eve| North|
+----------+-------+----------+-----------+------------+------+



In [57]:
orders_df.join(customer_df,"CustomerID","left").show()

+----------+-------+----------+-----------+------------+------+
|CustomerID|OrderID|   Product|SalesAmount|CustomerName|Region|
+----------+-------+----------+-----------+------------+------+
|       101|      1|    laptop|      50000|       Alice| North|
|       102|      2|     phone|      30000|         Bob| South|
|       103|      3|     table|      15000|     Charlie|  East|
|       104|      4|     chair|      10000|       David|  West|
|       105|      5|headphones|      20000|         Eve| North|
|       107|      7|    camera|      35000|        NULL|  NULL|
+----------+-------+----------+-----------+------------+------+



In [58]:
orders_df.join(customer_df,"CustomerID","right").show()

+----------+-------+----------+-----------+------------+------+
|CustomerID|OrderID|   Product|SalesAmount|CustomerName|Region|
+----------+-------+----------+-----------+------------+------+
|       101|      1|    laptop|      50000|       Alice| North|
|       102|      2|     phone|      30000|         Bob| South|
|       103|      3|     table|      15000|     Charlie|  East|
|       104|      4|     chair|      10000|       David|  West|
|       105|      5|headphones|      20000|         Eve| North|
|       108|   NULL|      NULL|       NULL|       Grace|  East|
+----------+-------+----------+-----------+------------+------+



In [59]:
orders_df.join(customer_df,"CustomerID","outer").show()

+----------+-------+----------+-----------+------------+------+
|CustomerID|OrderID|   Product|SalesAmount|CustomerName|Region|
+----------+-------+----------+-----------+------------+------+
|       101|      1|    laptop|      50000|       Alice| North|
|       102|      2|     phone|      30000|         Bob| South|
|       103|      3|     table|      15000|     Charlie|  East|
|       104|      4|     chair|      10000|       David|  West|
|       105|      5|headphones|      20000|         Eve| North|
|       107|      7|    camera|      35000|        NULL|  NULL|
|       108|   NULL|      NULL|       NULL|       Grace|  East|
+----------+-------+----------+-----------+------------+------+



In [60]:
orders_df.join(customer_df,"CustomerID","left_semi").show()

+----------+-------+----------+-----------+
|CustomerID|OrderID|   Product|SalesAmount|
+----------+-------+----------+-----------+
|       101|      1|    laptop|      50000|
|       102|      2|     phone|      30000|
|       103|      3|     table|      15000|
|       104|      4|     chair|      10000|
|       105|      5|headphones|      20000|
+----------+-------+----------+-----------+



In [61]:
orders_df.join(customer_df,"CustomerID","left_anti").show()

+----------+-------+-------+-----------+
|CustomerID|OrderID|Product|SalesAmount|
+----------+-------+-------+-----------+
|       107|      7| camera|      35000|
+----------+-------+-------+-----------+



In [62]:
df1 = spark.createDataFrame([(101,"North",100),
                             (101,"South",200),
                             (102,"East",150),
                             (103,"West",50)],
                            ["CustomerID", "Region", "SalesAmount"])

In [63]:
df2 = spark.createDataFrame([(101,"North","Gold"),
                             (101,"South","Silver"),    
                             (102,"East","Bronze"),
                             (103,"West","Platinum")],
                            ["CustomerID", "Region", "Tier"])

In [71]:
df1.join(
    df2,
    (df1["CustomerID"] == df2["CustomerID"]) &
    (df1["Region"] == df2["Region"]),
    "inner"
).show()

+----------+------+-----------+----------+------+--------+
|CustomerID|Region|SalesAmount|CustomerID|Region|    Tier|
+----------+------+-----------+----------+------+--------+
|       101| North|        100|       101| North|    Gold|
|       101| South|        200|       101| South|  Silver|
|       102|  East|        150|       102|  East|  Bronze|
|       103|  West|         50|       103|  West|Platinum|
+----------+------+-----------+----------+------+--------+



In [72]:
df1.join(
    df2,
    (df1.CustomerID == df2.CustomerID) &
    (df1.Region == df2.Region),
    "inner"
).show()

+----------+------+-----------+----------+------+--------+
|CustomerID|Region|SalesAmount|CustomerID|Region|    Tier|
+----------+------+-----------+----------+------+--------+
|       101| North|        100|       101| North|    Gold|
|       101| South|        200|       101| South|  Silver|
|       102|  East|        150|       102|  East|  Bronze|
|       103|  West|         50|       103|  West|Platinum|
+----------+------+-----------+----------+------+--------+



In [73]:
data = [(1,"Alice","North",1200),
        (2,"Bob","South",None),
        (3,"Charlie",None,1800),
        (4,"David","West",2000)
        ]

In [74]:
columns = ["CustomerID", "CustomerName", "Region", "SalesAmount"]

In [75]:
df = spark.createDataFrame(data,columns)

In [76]:
df.show()

+----------+------------+------+-----------+
|CustomerID|CustomerName|Region|SalesAmount|
+----------+------------+------+-----------+
|         1|       Alice| North|       1200|
|         2|         Bob| South|       NULL|
|         3|     Charlie|  NULL|       1800|
|         4|       David|  West|       2000|
+----------+------------+------+-----------+



In [78]:
df.filter(col("Region").isNull()).show()

+----------+------------+------+-----------+
|CustomerID|CustomerName|Region|SalesAmount|
+----------+------------+------+-----------+
|         3|     Charlie|  NULL|       1800|
+----------+------------+------+-----------+



In [79]:
df.filter(col("Region").isNotNull()).show()

+----------+------------+------+-----------+
|CustomerID|CustomerName|Region|SalesAmount|
+----------+------------+------+-----------+
|         1|       Alice| North|       1200|
|         2|         Bob| South|       NULL|
|         4|       David|  West|       2000|
+----------+------------+------+-----------+



In [80]:
df.columns

['CustomerID', 'CustomerName', 'Region', 'SalesAmount']

In [82]:
df.select("CustomerID").show()

+----------+
|CustomerID|
+----------+
|         1|
|         2|
|         3|
|         4|
+----------+



In [84]:
df.select(  [ count(when(col(c).isNull(),c)).alias(c+"_null_count") for c in df.columns ] ).show()

+---------------------+-----------------------+-----------------+----------------------+
|CustomerID_null_count|CustomerName_null_count|Region_null_count|SalesAmount_null_count|
+---------------------+-----------------------+-----------------+----------------------+
|                    0|                      0|                1|                     1|
+---------------------+-----------------------+-----------------+----------------------+



In [85]:
df.na.drop().show()

+----------+------------+------+-----------+
|CustomerID|CustomerName|Region|SalesAmount|
+----------+------------+------+-----------+
|         1|       Alice| North|       1200|
|         4|       David|  West|       2000|
+----------+------------+------+-----------+



In [87]:
df.dropna().show()

+----------+------------+------+-----------+
|CustomerID|CustomerName|Region|SalesAmount|
+----------+------------+------+-----------+
|         1|       Alice| North|       1200|
|         4|       David|  West|       2000|
+----------+------------+------+-----------+



In [88]:
df.filter(col("Region").isNotNull()).show()

+----------+------------+------+-----------+
|CustomerID|CustomerName|Region|SalesAmount|
+----------+------------+------+-----------+
|         1|       Alice| North|       1200|
|         2|         Bob| South|       NULL|
|         4|       David|  West|       2000|
+----------+------------+------+-----------+



In [89]:
df.na.drop(subset=["Region"]).show()

+----------+------------+------+-----------+
|CustomerID|CustomerName|Region|SalesAmount|
+----------+------------+------+-----------+
|         1|       Alice| North|       1200|
|         2|         Bob| South|       NULL|
|         4|       David|  West|       2000|
+----------+------------+------+-----------+



In [90]:
df.na.fill(0).show()

+----------+------------+------+-----------+
|CustomerID|CustomerName|Region|SalesAmount|
+----------+------------+------+-----------+
|         1|       Alice| North|       1200|
|         2|         Bob| South|          0|
|         3|     Charlie|  NULL|       1800|
|         4|       David|  West|       2000|
+----------+------------+------+-----------+



In [91]:
df.na.fill({"SalesAmount":0,"Region":"NA"})

DataFrame[CustomerID: bigint, CustomerName: string, Region: string, SalesAmount: bigint]

In [92]:
df.na.fill({"SalesAmount":0,"Region":"NA"}).show()

+----------+------------+------+-----------+
|CustomerID|CustomerName|Region|SalesAmount|
+----------+------------+------+-----------+
|         1|       Alice| North|       1200|
|         2|         Bob| South|          0|
|         3|     Charlie|    NA|       1800|
|         4|       David|  West|       2000|
+----------+------------+------+-----------+



In [94]:
df.select(
    "Region",
    coalesce(col("Region"),lit("NA")).alias("Region_Clean")
).show()

+------+------------+
|Region|Region_Clean|
+------+------------+
| North|       North|
| South|       South|
|  NULL|          NA|
|  West|        West|
+------+------------+



In [97]:
df.select("SalesAmount").show()

+-----------+
|SalesAmount|
+-----------+
|       1200|
|       NULL|
|       1800|
|       2000|
+-----------+



In [99]:
df.select(
    sum("SalesAmount").alias("Sum_Val")
    ,avg("SalesAmount").alias("Avg_Val")    
).show()

+-------+------------------+
|Sum_Val|           Avg_Val|
+-------+------------------+
|   5000|1666.6666666666667|
+-------+------------------+



In [100]:
df.select(
    sum("SalesAmount").alias("Sum_Val")
    ,avg("SalesAmount").alias("Avg_Val") 
    ,count("SalesAmount").alias("Cnt_Val")
    ,count("*").alias("Cnt_*")
).show()

+-------+------------------+-------+-----+
|Sum_Val|           Avg_Val|Cnt_Val|Cnt_*|
+-------+------------------+-------+-----+
|   5000|1666.6666666666667|      3|    4|
+-------+------------------+-------+-----+



In [ ]:
#   NULL != NULL